# 04 · Interpretabilidad

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tomas-pucv/Camanchaca-Predict/blob/main/notebooks/04_interpretability.ipynb)

**Propósito.** Explicar qué miran los modelos y contrastarlo con hipótesis declaradas **antes**
de ver los resultados:
- **H1.** DeiT-S usa más claves de campo lejano (cielo, horizonte, donde se acumula el *airlight*);
  ResNet-50 se apoya más en bordes locales cercanos.
- **H2.** Ambos usan en parte el atajo de contraste global (no solo la estructura en profundidad).
- **H3.** DeiT-S degrada menos con niebla densa y entre datasets (se evalúa en 03: por banda/fuente).

**Métodos.** Grad-CAM de regresión con signo explícito (−ŷ: evidencia de MENOR visibilidad;
+ŷ: de mayor) para ambos modelos y *attention rollout* para DeiT-S, sobre un conjunto FIJO de
ejemplos estratificado por fuente × banda (`<splits>/interpret_examples_val.csv` en desarrollo;
`interpret_examples.csv` de test solo con `--final`; ambos se versionan). Contrafactual:
niebla con profundidad vs niebla uniforme emparejada en canal oscuro.

**Tiempo esperado (T4).** ~10–15 min.
**Salidas.** `reports/figures/interpret_*.png`, `reports/tables/attribution.md`,
`reports/tables/counterfactual.md`.

## Configuración común
Misma celda en todos los notebooks (detalles en `00_setup_colab`). En **Colab**: monta Drive,
fija `CAMANCHACA_DATA=/content/data` y `CAMANCHACA_RUNS=/content/drive/MyDrive/camanchaca/runs`,
clona/actualiza el repo en la referencia `REF`, instala el paquete (`pip install -e ".[demo,data]"`,
**sin** reinstalar torch) y desempaqueta los `.tar` de datos de Drive (salvo `NEEDS_DATA = False`). **Fuera de Colab** (`SMOKE=True`)
usa datos sintéticos y pesos aleatorios generados por `camanchaca smoke`: sirve para verificar que
todo corre, **sus números no son resultados**.

In [ ]:
# Configuración común (idéntica en los notebooks 00-06)
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = not IN_COLAB   # True: datos sintéticos + pesos aleatorios (solo verifica que todo corre)
REF = "main"           # rama/tag/commit del repo; para la entrega final usar "v1.0-final"
REPO_URL = "https://github.com/Tomas-pucv/Camanchaca-Predict.git"
DRIVE_DIR = Path("/content/drive/MyDrive/camanchaca")   # tars de datos, corridas, pesos y credenciales

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["CAMANCHACA_DATA"] = "/content/data"
    os.environ["CAMANCHACA_RUNS"] = str(DRIVE_DIR / "runs")
    REPO = Path("/content/Camanchaca-Predict")
    if not REPO.exists():
        !git clone -q {REPO_URL} {REPO}
    !git -C {REPO} fetch -q --all --tags && git -C {REPO} checkout -q {REF} && (git -C {REPO} pull -q --ff-only 2>/dev/null || true)
    %cd {REPO}
    !pip install -q -e ".[demo,data]"   # NO reinstala torch/torchvision: se usan los de Colab
    if not SMOKE and globals().get("NEEDS_DATA", True):
        for tar in sorted((DRIVE_DIR / "data").glob("*.tar")):   # imágenes preprocesadas (notebook 01)
            !camanchaca unpack --tar {tar}
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    %cd {REPO}
    if shutil.which("camanchaca") is None:
        print("AVISO: falta el comando `camanchaca`: ejecutar  pip install -e '.[dev,demo]'")

WORK = Path(os.environ.get("CAMANCHACA_SMOKE_DIR", str(REPO / "local_workspace" / "nb_smoke")))
if SMOKE:
    os.environ.update(CAMANCHACA_DATA=str(WORK / "data"), CAMANCHACA_RUNS=str(WORK / "runs"),
                      CAMANCHACA_REPORTS=str(WORK / "reports"), HF_HUB_OFFLINE="1")
    if not (WORK / "smoke.yaml").exists():   # ~4-6 min en CPU, una sola vez
        !camanchaca smoke --workdir {WORK}

sys.path.insert(0, str(REPO / "src"))   # importable sin reiniciar el kernel tras `pip install -e`
from camanchaca.config import load_yaml
from camanchaca.paths import data_root, reports_dir, runs_root

EXP = str(WORK / "smoke.yaml") if SMOKE else "configs/experiment/main.yaml"
EXP_NAME = load_yaml(EXP)["name"]
print(f"Colab={IN_COLAB} | SMOKE={SMOKE} | EXP={EXP}")
print(f"datos={data_root()} | corridas={runs_root()} | reportes={reports_dir()}")

## 1. Grad-CAM y attention rollout
Mismos ejemplos para ambos modelos. Con `--final` se usan ejemplos de test (solo en la ronda final).

In [ ]:
FINAL = SMOKE   # True solo en la ronda final
FINAL_FLAG = "--final" if FINAL else ""
!camanchaca interpret --exp {EXP} --seed 0 --n-per-cell 2 {FINAL_FLAG}

In [ ]:
from IPython.display import Image, Markdown, display
TABLES, FIGS = reports_dir() / "tables", reports_dir() / "figures"
for p in sorted(FIGS.glob("interpret_*.png")):
    display(Image(filename=str(p), width=900))
if (TABLES / "attribution.md").exists(): display(Markdown((TABLES / "attribution.md").read_text()))

## 2. Contrafactual: niebla con profundidad vs niebla uniforme
Si la predicción casi no cambia al reemplazar la niebla dependiente de la distancia por una
uniforme con el mismo canal oscuro, el modelo usa sobre todo el contraste global (atajo, H2).

In [ ]:
!camanchaca counterfactual --exp {EXP} --n 200 {FINAL_FLAG}
if (TABLES / "counterfactual.md").exists(): display(Markdown((TABLES / "counterfactual.md").read_text()))

## 3. Preguntas guía (responder con números y figuras en el informe)
- **H1.** En `attribution.md`, ¿la fracción de atribución en el tercil **lejano** y en **cielo** es
  mayor en DeiT-S que en ResNet-50? ¿Los mapas −ŷ de DeiT cubren horizonte/cielo y los de ResNet
  bordes cercanos (calzada, vehículos)? ¿Coincide el rollout con el Grad-CAM de DeiT?
- **H2.** En `counterfactual.md`, ¿cuánto cambia V̂ entre niebla con profundidad y uniforme? ¿La
  sonda de fuente (`probe_source.md`, notebook 03) predice el dataset con alta exactitud?
- **H3.** En `metrics_<set>.md` y las figuras por banda/fuente (03), ¿DeiT-S pierde menos en la
  banda Crítico y en HazeRD/O-HAZE?
- ¿Los errores **falso-seguro** (V real < 100 m, V̂ ≥ 100 m) muestran evidencia +ŷ en estructuras
  nítidas cercanas?

Si una hipótesis no se cumple, se reporta igual: es un resultado.